In [ ]:
'''
带表格全功率
'''
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.cuda.amp import GradScaler, autocast
import os
import time
import numpy as np
import random
import json
import csv
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, precision_score, recall_score

# 导入您定义的模型

from models.soilnetgraph import SoilNetHybrid, build_soilnet_hybrid
# 设置随机种子
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 时间跟踪器
class TimeTracker:
    def __init__(self):
        self.epoch_times = []
        self.batch_times = []
        self.epoch_start = None
        self.batch_start = None
    
    def start_epoch(self):
        self.epoch_start = time.time()
    
    def end_epoch(self):
        epoch_time = time.time() - self.epoch_start
        self.epoch_times.append(epoch_time)
        return epoch_time
    
    def start_batch(self):
        self.batch_start = time.time()
    
    def end_batch(self):
        batch_time = time.time() - self.batch_start
        self.batch_times.append(batch_time)
        return batch_time
    
    def get_stats(self):
        return {
            'total_epoch_time': sum(self.epoch_times),
            'avg_epoch_time': np.mean(self.epoch_times),
            'median_epoch_time': np.median(self.epoch_times),
            'min_epoch_time': min(self.epoch_times),
            'max_epoch_time': max(self.epoch_times),
            'avg_batch_time': np.mean(self.batch_times) if self.batch_times else 0,
            'total_batch_time': sum(self.batch_times) if self.batch_times else 0
        }

# 混合数据增强
class MixupCutmix:
    def __init__(self, mixup_alpha=0.8, cutmix_alpha=1.0, switch_prob=0.5):
        self.mixup_beta = torch.distributions.Beta(mixup_alpha, mixup_alpha)
        self.cutmix_beta = torch.distributions.Beta(cutmix_alpha, cutmix_alpha)
        self.switch_prob = switch_prob

    def __call__(self, x, y):
        if random.random() < self.switch_prob:  # CutMix
            lam = self.cutmix_beta.sample().item()
            bbx1, bby1, bbx2, bby2 = self.rand_bbox(x.size(), lam)
            mixed_x = x.clone()
            mixed_x[:, :, bbx1:bbx2, bby1:bby2] = x.flip(0)[:, :, bbx1:bbx2, bby1:bby2]
            lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))
            return mixed_x, (y, y.flip(0), lam)
        else:  # Mixup
            lam = self.mixup_beta.sample().item()
            mixed_x = lam * x + (1 - lam) * x.flip(0)
            return mixed_x, (y, y.flip(0), lam)

    def rand_bbox(self, size, lam):
        W, H = size[2], size[3]
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)
        return bbx1, bby1, bbx2, bby2

# 随机擦除增强
class RandomErasing:
    def __init__(self, p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value=0):
        self.p = p
        self.scale = scale
        self.ratio = ratio
        self.value = value

    def __call__(self, img):
        if random.random() > self.p:
            return img
        
        C, H, W = img.shape
        area = H * W
        
        for _ in range(10):
            erase_area = random.uniform(*self.scale) * area
            aspect_ratio = random.uniform(*self.ratio)
            
            h = int(round(np.sqrt(erase_area * aspect_ratio)))
            w = int(round(np.sqrt(erase_area / aspect_ratio)))
            
            if h < H and w < W:
                i = random.randint(0, H - h)
                j = random.randint(0, W - w)
                img[:, i:i+h, j:j+w] = self.value
                return img
        
        return img

# 训练函数
def train_one_epoch(model, train_loader, optimizer, criterion, scaler, device, 
                   mixup_cutmix=None, mix_prob=0.8, time_tracker=None):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []
    
    pbar = tqdm(train_loader, desc='Training')
    for batch_idx, (inputs, targets) in enumerate(pbar):
        if time_tracker:
            time_tracker.start_batch()
        
        inputs, targets = inputs.to(device), targets.to(device)
        
        # 应用混合增强
        if mixup_cutmix and random.random() < mix_prob:
            mixed_inputs, mixed_targets = mixup_cutmix(inputs, targets)
            y_a, y_b, lam = mixed_targets
            inputs = mixed_inputs
        else:
            y_a = targets
            y_b = targets
            lam = 1.0
        
        optimizer.zero_grad()
        
        # 混合精度训练
        with autocast():
            outputs = model(inputs)
            loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # 计算指标
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += (lam * predicted.eq(y_a).sum().float() + 
                   (1 - lam) * predicted.eq(y_b).sum().float()).item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
        
        # 更新进度条
        avg_loss = total_loss / (batch_idx + 1)
        acc = 100. * correct / total
        pbar.set_postfix({'Loss': f'{avg_loss:.4f}', 'Acc': f'{acc:.2f}%'})
        
        if time_tracker:
            time_tracker.end_batch()
    
    # 计算训练集F1分数
    train_f1 = f1_score(all_targets, all_preds, average='macro') if all_targets else 0.0
    return {
        'train_loss': total_loss / len(train_loader),
        'train_acc': correct / total,
        'train_f1': train_f1
    }

# 验证函数
@torch.no_grad()
def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []
    
    pbar = tqdm(val_loader, desc='Validation')
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
        
        # 更新进度条
        avg_loss = total_loss / (len(val_loader) if pbar.n == 0 else pbar.n)
        acc = 100. * correct / total
        pbar.set_postfix({'Loss': f'{avg_loss:.4f}', 'Acc': f'{acc:.2f}%'})
    
    # 计算验证集指标
    accuracy = accuracy_score(all_targets, all_preds)
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    class_report = classification_report(all_targets, all_preds, output_dict=True, zero_division=0)
    cm = confusion_matrix(all_targets, all_preds)
    
    return {
        'val_loss': total_loss / len(val_loader),
        'val_acc': accuracy,
        'val_f1': f1,
        'val_precision': precision,
        'val_recall': recall,
        'classification_report': class_report,
        'confusion_matrix': cm
    }

# 主训练函数
def main():
    # 配置参数
    config = {
        'data_path': 'soil',  # 数据集路径
        'num_classes': 6,      # 类别数
        'epochs': 300,         # 训练轮数
        'batch_size': 32,      # 批大小
        'lr': 3e-4,            # 学习率
        'weight_decay': 1e-4,  # 权重衰减
        'seed': 42,            # 随机种子
        'mixup_alpha': 0.8,    # Mixup参数
        'cutmix_alpha': 1.0,   # CutMix参数
        'mix_prob': 0.8,       # 应用混合增强的概率
        'erase_prob': 0.25,    # 随机擦除概率
        'save_dir': 'results', # 结果保存目录
    }
    
    # 设置随机种子
    set_seed(config['seed'])
    
    # 创建结果目录
    os.makedirs(config['save_dir'], exist_ok=True)
    
    # 数据增强
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        RandomErasing(p=config['erase_prob'])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # 加载数据集
    train_dataset = datasets.ImageFolder(
        os.path.join(config['data_path'], 'train'), 
        transform=train_transform
    )
    val_dataset = datasets.ImageFolder(
        os.path.join(config['data_path'], 'val'), 
        transform=val_transform
    )
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config['batch_size'], 
        shuffle=True, 
        num_workers=4,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=config['batch_size'], 
        num_workers=4,
        pin_memory=True
    )
    
    # 初始化设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # 初始化模型
    model = build_soilnet_hybrid(num_classes=config['num_classes']).to(device)
    print(f"Model created with {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")
    
    # 初始化混合增强
    mixup_cutmix = MixupCutmix(
        mixup_alpha=config['mixup_alpha'],
        cutmix_alpha=config['cutmix_alpha']
    )
    
    # 损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['lr'], 
        weight_decay=config['weight_decay']
    )
    scaler = GradScaler()
    
    # 学习率调度器
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=config['epochs']
    )
    
    # 时间跟踪器
    time_tracker = TimeTracker()
    
    # 训练历史记录
    history = {
        'epoch': [],
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': [],
        'train_f1': [],
        'val_f1': [],
        'val_precision': [],
        'val_recall': [],
        'lr': []
    }
    
    # 创建CSV记录文件
    csv_path = os.path.join(config['save_dir'], 'training_metrics.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            'epoch', 'train_loss', 'val_loss', 
            'train_acc', 'val_acc', 'train_f1', 'val_f1',
            'val_precision', 'val_recall', 'lr'
        ])
    
    # 最佳准确率
    best_val_acc = 0.0
    
    # 训练循环
    for epoch in range(1, config['epochs'] + 1):
        print(f"\nEpoch {epoch}/{config['epochs']}")
        time_tracker.start_epoch()
        
        # 训练
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler, device,
            mixup_cutmix, config['mix_prob'], time_tracker
        )
        
        # 验证
        val_metrics = validate(model, val_loader, criterion, device)
        
        # 更新学习率
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        
        # 记录历史
        history['epoch'].append(epoch)
        history['train_loss'].append(train_metrics['train_loss'])
        history['val_loss'].append(val_metrics['val_loss'])
        history['train_acc'].append(train_metrics['train_acc'])
        history['val_acc'].append(val_metrics['val_acc'])
        history['train_f1'].append(train_metrics['train_f1'])
        history['val_f1'].append(val_metrics['val_f1'])
        history['val_precision'].append(val_metrics['val_precision'])
        history['val_recall'].append(val_metrics['val_recall'])
        history['lr'].append(current_lr)
        
        # 保存到CSV
        with open(csv_path, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch, train_metrics['train_loss'], val_metrics['val_loss'],
                train_metrics['train_acc'], val_metrics['val_acc'], 
                train_metrics['train_f1'], val_metrics['val_f1'],
                val_metrics['val_precision'], val_metrics['val_recall'],
                current_lr
            ])
        
        # 打印指标
        epoch_time = time_tracker.end_epoch()
        print(f"Epoch {epoch} completed in {epoch_time:.2f}s")
        print(f"Train Loss: {train_metrics['train_loss']:.4f} | Val Loss: {val_metrics['val_loss']:.4f}")
        print(f"Train Acc: {train_metrics['train_acc']*100:.2f}% | Val Acc: {val_metrics['val_acc']*100:.2f}%")
        print(f"Val F1: {val_metrics['val_f1']:.4f} | Val Precision: {val_metrics['val_precision']:.4f} | Val Recall: {val_metrics['val_recall']:.4f}")
        print(f"Learning Rate: {current_lr:.2e}")
        
        # 保存最佳模型
        if val_metrics['val_acc'] > best_val_acc:
            best_val_acc = val_metrics['val_acc']
            model_save_path = os.path.join(config['save_dir'], f'best_model_acc_{best_val_acc*100:.2f}.pth')
            torch.save(model.state_dict(), model_save_path)
            print(f"Saved best model with val acc: {best_val_acc*100:.2f}%")
            
            # 保存分类报告和混淆矩阵
            with open(os.path.join(config['save_dir'], 'best_classification_report.json'), 'w') as f:
                json.dump(val_metrics['classification_report'], f, indent=4)
            
            np.savetxt(os.path.join(config['save_dir'], 'best_confusion_matrix.csv'), 
                      val_metrics['confusion_matrix'], delimiter=',', fmt='%d')
    
    # 保存最终模型
    torch.save(model.state_dict(), os.path.join(config['save_dir'], 'final_model.pth'))
    
    # 保存完整历史
    with open(os.path.join(config['save_dir'], 'training_history.json'), 'w') as f:
        json.dump(history, f, indent=4)
    
    # 打印时间统计
    time_stats = time_tracker.get_stats()
    print("\nTraining completed!")
    print(f"Total training time: {time_stats['total_epoch_time']:.2f}s")
    print(f"Average epoch time: {time_stats['avg_epoch_time']:.2f}s")
    print(f"Best validation accuracy: {best_val_acc*100:.2f}%")
    
    # 保存时间统计
    with open(os.path.join(config['save_dir'], 'time_stats.json'), 'w') as f:
        json.dump(time_stats, f, indent=4)

if __name__ == "__main__":
    main()


In [ ]:
'''
去除wandb  残差50蒸馏
'''





import os
import time
import random
import itertools
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from ptflops import get_model_complexity_info
from models.tiny_vit import tiny_vit_5m_224, tiny_vit_11m_224, tiny_vit_21m_224, tiny_vit_21m_384, tiny_vit_21m_512

# ----------------- 数据增强 -----------------

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.2)),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ----------------- 可视化工具 -----------------
class Visualizer:
    @staticmethod
    def cnf_matrix_plotter(cm, classes, cmap=plt.cm.Blues, filename='confusion_matrix.pdf'):
        plt.figure(figsize=(8, 6))
        plt.imshow(cm, interpolation='nearest', cmap=cmap)
        plt.title('Confusion Matrix', fontsize=14, pad=20)
        plt.colorbar()

        tick_marks = np.arange(len(classes))
        plt.xticks(tick_marks, classes, rotation=45, fontsize=12)
        plt.yticks(tick_marks, classes, rotation=45, fontsize=12)

        thresh = cm.max() / 2.
        for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
            plt.text(j, i, f"{cm[i, j]}",
                     horizontalalignment="center",
                     verticalalignment="center",
                     color="white" if cm[i, j] > thresh else "black",
                     fontsize=12)

        plt.ylabel('True label', fontsize=14)
        plt.xlabel('Predicted label', fontsize=14)
        plt.tight_layout()
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.close()

    @staticmethod 
    def plot_tsne(features, labels, class_names, filename='tsne.pdf'):
        tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=300)
        features_tsne = tsne.fit_transform(features)

        plt.figure(figsize=(10, 8))
        colors = list(mcolors.TABLEAU_COLORS.values())

        for i, class_name in enumerate(class_names):
            mask = labels == i
            plt.scatter(features_tsne[mask, 0], features_tsne[mask, 1],
                        c=[colors[i]], label=class_name, s=50, alpha=0.6)

        plt.xticks(fontsize=12)
        plt.yticks(fontsize=12)
        plt.xlabel('t-SNE 1', fontsize=14)
        plt.ylabel('t-SNE 2', fontsize=14)
        plt.legend(loc='upper right', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.close()

    @staticmethod
    def plot_training_metrics(history, filename='training_metrics.pdf'):
        epochs = range(1, len(history['train_loss']) + 1)

        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

        ax1.plot(epochs, history['train_loss'], 'b-', label='Train')
        ax1.plot(epochs, history['val_loss'], 'r-', label='Validation')
        ax1.set_title('Loss Curve')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.plot(epochs, history['train_acc'], 'b-', label='Train')
        ax2.plot(epochs, history['val_acc'], 'r-', label='Validation')
        ax2.set_title('Accuracy Curve')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        ax3.plot(epochs, history['train_f1'], 'b-', label='Train')
        ax3.plot(epochs, history['val_f1'], 'r-', label='Validation')
        ax3.set_title('F1 Score Curve')
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('F1 Score')
        ax3.legend()
        ax3.grid(True, alpha=0.3)

        ax4.plot(epochs, history['lr'], 'g-', label='Learning Rate')
        ax4.set_title('Learning Rate Schedule')
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('Learning Rate')
        ax4.legend()
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.close()

# ----------------- 知识蒸馏损失 -----------------
class DistillationLoss(nn.Module):
    def __init__(self, temperature=3.0, alpha=0.7, mode='logits'):
        super().__init__()
        self.T = temperature
        self.alpha = alpha
        self.mode = mode
        self.ce_loss = nn.CrossEntropyLoss(label_smoothing=0.1)
        
    def forward(self, student_outputs, teacher_outputs, targets):
        # 硬目标损失
        hard_loss = self.ce_loss(student_outputs['logits'], targets)
        
        # 软目标损失
        if self.mode == 'logits':
            soft_loss = F.kl_div(
                F.log_softmax(student_outputs['logits']/self.T, dim=1),
                F.softmax(teacher_outputs['logits']/self.T, dim=1),
                reduction='batchmean'
            ) * (self.T ** 2)
        else:  # 特征蒸馏
            soft_loss = 0
            for layer in student_outputs['features']:
                s_feat = student_outputs['features'][layer]
                t_feat = teacher_outputs['features'][layer]
                soft_loss += F.mse_loss(s_feat, t_feat)
        
        return self.alpha * soft_loss + (1 - self.alpha) * hard_loss

# ----------------- TinyViT模型包装器（支持特征提取）-----------------
class TinyViTWrapper(nn.Module):
    def __init__(self, model_name='tiny_vit_5m_224', num_classes=6, pretrained=True, 
                 pretrained_path=None, distill_layers=None):
        super().__init__()
        self.model = tiny_vit_5m_224(pretrained=pretrained, num_classes=num_classes)
        
        if pretrained_path:
            state_dict = torch.load(pretrained_path, weights_only=True)
            self.model.load_state_dict(state_dict, strict=False)
            
        self.distill_layers = distill_layers or [
            f'layers.{len(self.model.layers)-3}.blocks.{i}' 
            for i in range(self.model.depths[-3])
        ]
        
        self.original_head = self.model.head
        self.model.head = nn.Linear(self.model.head.in_features, num_classes)
        
        # Get actual resolutions from model
        self.layer_resolutions = self._get_actual_resolutions()

    def _get_actual_resolutions(self):
        """Get actual resolutions from model layers"""
        resolutions = []
        # Get patch embed resolution
        H = self.model.patch_embed.patches_resolution[0]
        W = self.model.patch_embed.patches_resolution[1]
        resolutions.append((H, W))
        
        # Get resolutions from layers
        for layer in self.model.layers:
            if hasattr(layer, 'blocks'):
                if len(layer.blocks) > 0 and hasattr(layer.blocks[0], 'input_resolution'):
                    H, W = layer.blocks[0].input_resolution
                    resolutions.append((H, W))
                else:
                    resolutions.append(resolutions[-1])
            else:
                resolutions.append(resolutions[-1])
        
        return resolutions[:len(self.model.layers)+1]

    def forward(self, x, return_features=False):
        if not return_features:
            return {
                'logits': self.model(x),
                'features': None
            }
            
        features = {}
        B = x.shape[0]
        
        # 1. Patch embedding
        x = self.model.patch_embed(x)
        C, H, W = x.shape[1], x.shape[2], x.shape[3]
        
        # Check initial resolution
        if (H, W) != self.layer_resolutions[0]:
            raise ValueError(
                f"Initial resolution mismatch: expected {self.layer_resolutions[0]}, got {(H, W)}"
            )
        
        # Convert to sequence format (B, L, C)
        x = x.flatten(2).permute(0, 2, 1)  # (B, C, H*W) -> (B, H*W, C)
        
        # 2. Process through layers
        for layer_idx, (layer, (H_layer, W_layer)) in enumerate(zip(self.model.layers, self.layer_resolutions[1:])):
            # Handle downsampling layers
            if hasattr(layer, 'downsample') and layer.downsample is not None:
                x = layer.downsample(x)
                H, W = H_layer, W_layer
                continue
            
            if hasattr(layer, 'blocks'):
                for block_idx, block in enumerate(layer.blocks):
                    # Ensure input shape is correct
                    if hasattr(block, 'input_resolution'):
                        expected_H, expected_W = block.input_resolution
                        current_L = x.shape[1]
                        if current_L != expected_H * expected_W:
                            # Reshape to spatial format for interpolation
                            x = x.permute(0, 2, 1).reshape(B, -1, H, W)
                            x = F.interpolate(x, size=(expected_H, expected_W), mode='nearest')
                            x = x.flatten(2).permute(0, 2, 1)
                            H, W = expected_H, expected_W
                    
                    # Save current shape for recovery
                    prev_shape = x.shape
                    
                    # Execute block forward pass
                    x = block(x)
                    
                    # Feature collection
                    name = f'layers.{layer_idx}.blocks.{block_idx}'
                    if name in self.distill_layers:
                        features[name] = x.mean(dim=1)  # (B, C)
                    
                    # Update channel dimension
                    C = x.shape[-1]
            else:
                x = layer(x)
                if f'layers.{layer_idx}' in self.distill_layers:
                    features[f'layers.{layer_idx}'] = x.mean(dim=1)
        
        # 3. Final classification
        x = x.mean(1)  # (B, C)
        x = self.model.norm_head(x)
        logits = self.model.head(x)
        
        return {
            'logits': logits,
            'features': features
        }


# ----------------- ResNet教师模型包装器 -----------------
class ResNetTeacher(nn.Module):
    def __init__(self, model_name='resnet50', num_classes=6):
        super().__init__()
        self.model = getattr(models, model_name)(pretrained=True)
        
        # 替换最后的全连接层
        in_features = self.model.fc.in_features
        self.model.fc = nn.Linear(in_features, num_classes)
        
        # 特征提取层
        self.feature_layers = ['layer1', 'layer2']
        
    def forward(self, x, return_features=False):
        if return_features:
            features = {}
            
            x = self.model.conv1(x)
            x = self.model.bn1(x)
            x = self.model.relu(x)
            x = self.model.maxpool(x)

            x = self.model.layer1(x)
            features['layer1'] = F.adaptive_avg_pool2d(x, (1, 1)).squeeze()
            
            x = self.model.layer2(x)
            features['layer2'] = F.adaptive_avg_pool2d(x, (1, 1)).squeeze()
            
            x = self.model.layer3(x)
            x = self.model.layer4(x)
            
            x = self.model.avgpool(x)
            x = torch.flatten(x, 1)
            x = self.model.fc(x)
            
            return {
                'logits': x,
                'features': features
            }
        return {
            'logits': self.model(x),
            'features': None
        }

# ----------------- 训练时间记录器 -----------------
class TimeTracker:
    def __init__(self):
        self.epoch_times = []
        self.batch_times = []
        self.start_time = None

    def epoch_start(self):
        self.start_time = time.time()

    def epoch_end(self):
        epoch_time = time.time() - self.start_time
        self.epoch_times.append(epoch_time)
        return epoch_time

    def batch_end(self):
        self.batch_times.append(time.time() - self.start_time)

    def get_stats(self):
        return {
            'total_time': sum(self.epoch_times),
            'avg_epoch_time': np.mean(self.epoch_times),
            'avg_batch_time': np.mean(self.batch_times) if self.batch_times else 0,
            'epoch_times': self.epoch_times
        }

# ----------------- 训练函数 -----------------
def train_one_epoch(student, teacher, loader, optimizer, scaler, criterion, epoch, time_tracker, config):
    student.train()
    total_loss = 0.0
    preds, labels = [], []

    time_tracker.epoch_start()
    pbar = tqdm(loader, desc=f'Epoch {epoch} Training')

    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=True):
            # 教师模型推理
            with torch.no_grad():
                teacher_outputs = teacher(images, return_features=True)
            
            # 学生模型推理
            student_outputs = student(images, return_features=True)
            
            # 计算蒸馏损失
            loss = criterion(student_outputs, teacher_outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        time_tracker.batch_end()

        total_loss += loss.item()
        batch_preds = torch.argmax(student_outputs['logits'], 1).cpu().numpy()
        batch_labels = targets.cpu().numpy()
        preds.extend(batch_preds)
        labels.extend(batch_labels)

        pbar.set_postfix({'Loss': loss.item()})

    epoch_time = time_tracker.epoch_end()
    return {
        'train_loss': total_loss / len(loader),
        'train_acc': accuracy_score(labels, preds),
        'train_f1': f1_score(labels, preds, average='macro'),
        'epoch_time': epoch_time
    }

# ----------------- 验证函数 -----------------
@torch.no_grad()
def evaluate(model, loader, criterion, epoch, is_best=False):
    model.eval()
    total_loss = 0.0
    preds, labels, features = [], [], []

    for images, targets in tqdm(loader, desc='Evaluating'):
        images = images.to(device)
        targets = targets.to(device)

        with torch.cuda.amp.autocast(enabled=True):
            outputs = model(images, return_features=True)
            loss = criterion(outputs, None, targets) 

        total_loss += loss.item()
        preds.extend(torch.argmax(outputs['logits'], 1).cpu().numpy())
        labels.extend(targets.cpu().numpy())
        if outputs['features'] is not None:
            features.extend([f.mean(dim=0).cpu().numpy() for f in outputs['features'].values()])

    metrics = {
        'val_loss': total_loss / len(loader),
        'val_acc': accuracy_score(labels, preds),
        'val_precision': precision_score(labels, preds, average='macro'),
        'val_recall': recall_score(labels, preds, average='macro'),
        'val_f1': f1_score(labels, preds, average='macro')
    }

    if is_best and features:
        idx_to_labels = np.load('idx_to_labels.npy', allow_pickle=True).item()
        class_names = [idx_to_labels[i] for i in range(len(idx_to_labels))]
        
        features = np.array(features)
        Visualizer.plot_tsne(features, np.array(labels), class_names, 
                 filename=f'checkpoint/best_tsne_epoch{epoch}.pdf')

        cm = confusion_matrix(labels, preds)
        Visualizer.cnf_matrix_plotter(cm, class_names, filename=f'checkpoint/best_cm_epoch{epoch}.pdf')
        print("\nClassification Report:")
        print(classification_report(labels, preds, target_names=class_names, digits=4))

    return metrics

# ----------------- 主函数 -----------------
def main():
    # 初始化配置
    config = {
        'model_name': 'tiny_vit_5m_224',
        'teacher_model': 'resnet50',
        'num_classes': 6,
        'pretrained_path': 'tiny_vit_5m_22kto1k_distill.pth',
        'distill_mode': 'features',  # 'logits' or 'features'
        'temperature': 3.0,
        'alpha': 0.7,
        'epochs': 80,
        'batch_size': 64,
        'lr': 2e-4,
        'weight_decay': 0.05,
        'seed': 42
    }
    
    time_tracker = TimeTracker()

    # 设置随机种子
    random.seed(config['seed'])
    np.random.seed(config['seed'])
    torch.manual_seed(config['seed'])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config['seed'])

    # 数据加载
    train_dataset = datasets.ImageFolder('soil/train', train_transform)
    test_dataset = datasets.ImageFolder('soil/val', test_transform)

    idx_to_labels = {v: k for k, v in train_dataset.class_to_idx.items()}
    np.save('idx_to_labels.npy', idx_to_labels)

    train_loader = DataLoader(
        train_dataset, batch_size=config['batch_size'], shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=config['batch_size'],
        num_workers=4, pin_memory=True, persistent_workers=True
    )

    # 初始化模型
    student = TinyViTWrapper(
        model_name=config['model_name'],
        num_classes=len(idx_to_labels),
        pretrained_path=config['pretrained_path']
    ).to(device)
    
    teacher = ResNetTeacher(
        model_name=config['teacher_model'],
        num_classes=len(idx_to_labels)
    ).to(device).eval()
    
    # 冻结教师模型
    for param in teacher.parameters():
        param.requires_grad = False

    # 计算模型统计信息
    macs, params = get_model_complexity_info(student, (3, 224, 224), as_strings=False)
    model_stats = {
        'params(M)': params / 1e6,
        'FLOPs(G)': macs / 1e9,
        'MACs(G)': macs / 1e9 * 2
    }
    print("\nStudent Model Analysis:")
    print(f"Parameters: {model_stats['params(M)']:.2f}M")
    print(f"FLOPs: {model_stats['FLOPs(G)']:.2f}G")

    # 训练准备
    criterion = DistillationLoss(
        temperature=config['temperature'],
        alpha=config['alpha'],
        mode=config['distill_mode']
    )
    optimizer = torch.optim.AdamW(
        student.parameters(), 
        lr=config['lr'], 
        weight_decay=config['weight_decay']
    )
    scaler = GradScaler()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, 
        T_0=10, 
        T_mult=2
    )

    # 训练循环
    best_acc = 0.0
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'train_f1': [], 'val_f1': [],
        'lr': [], 'epoch_times': []
    }

    for epoch in range(1, config['epochs'] + 1):
        train_metrics = train_one_epoch(
            student, teacher, train_loader, 
            optimizer, scaler, criterion, 
            epoch, time_tracker, config
        )
        val_metrics = evaluate(student, test_loader, criterion, epoch)

        # 更新最佳模型
        if val_metrics['val_acc'] > best_acc:
            best_acc = val_metrics['val_acc']
            torch.save(student.state_dict(), f'checkpoint/best_model_{best_acc:.3f}.pth')
            _ = evaluate(student, test_loader, criterion, epoch, is_best=True)
            print(f'New best model saved with acc {best_acc:.3f}')

        # 更新历史记录
        for k in history:
            if k in train_metrics: history[k].append(train_metrics[k])
            elif k in val_metrics: history[k].append(val_metrics[k])
            elif k == 'lr': history[k].append(optimizer.param_groups[0]['lr'])

        scheduler.step()

        # 打印当前epoch信息
        print(f"Epoch {epoch}/{config['epochs']}: "
              f"Train Loss: {train_metrics['train_loss']:.4f}, "
              f"Train Acc: {train_metrics['train_acc']:.4f}, "
              f"Val Loss: {val_metrics['val_loss']:.4f}, "
              f"Val Acc: {val_metrics['val_acc']:.4f}, "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")

    # 保存最终模型
    torch.save(student.state_dict(), 'checkpoint/final_model.pth')
    
    # 训练结束统计
    training_stats = {
        'total_time': time_tracker.get_stats()['total_time'],
        'avg_epoch_time': time_tracker.get_stats()['avg_epoch_time'],
        'best_val_acc': best_acc,
        **model_stats
    }
    np.savez('checkpoint/final_stats.npz', **training_stats)
    
    # 绘制训练曲线
    Visualizer.plot_training_metrics(history, filename='checkpoint/training_metrics.pdf')

    print("\nTraining Completed with Stats:")
    print(f"Total Training Time: {training_stats['total_time']/3600:.2f} hours")
    print(f"Average Epoch Time: {training_stats['avg_epoch_time']:.2f} seconds")
    print(f"Best Validation Accuracy: {best_acc:.4f}")

if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = True
    os.makedirs('checkpoint', exist_ok=True)
    main()


In [ ]:
'''
swin pretrain
'''
import os
import time
import random
import itertools
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import wandb
from ptflops import get_model_complexity_info
from models.tiny_vit import tiny_vit_5m_224, tiny_vit_11m_224, tiny_vit_21m_224, tiny_vit_21m_384, tiny_vit_21m_512

# ----------------- 数据增强 -----------------
class AddSoilNoise:
    def __call__(self, tensor):
        noise = torch.randn_like(tensor) * 0.03 * random.random()
        return torch.clamp(tensor + noise, 0, 1)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    AddSoilNoise(),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.2)),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ----------------- 可视化工具 -----------------
class Visualizer:
    @staticmethod
    def cnf_matrix_plotter(cm, classes, cmap=plt.cm.Blues, filename='confusion_matrix.pdf'):
        plt.figure(figsize=(8, 6))
        plt.imshow(cm, interpolation='nearest', cmap=cmap)
        plt.title('Confusion Matrix', fontsize=14, pad=20)
        plt.colorbar()

        tick_marks = np.arange(len(classes))
        plt.xticks(tick_marks, classes, rotation=45, fontsize=12)
        plt.yticks(tick_marks, classes, rotation=45, fontsize=12)

        thresh = cm.max() / 2.
        for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
            plt.text(j, i, f"{cm[i, j]}",
                     horizontalalignment="center",
                     verticalalignment="center",
                     color="white" if cm[i, j] > thresh else "black",
                     fontsize=12)

        plt.ylabel('True label', fontsize=14)
        plt.xlabel('Predicted label', fontsize=14)
        plt.tight_layout()
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.close()

    @staticmethod 
    def plot_tsne(features, labels, class_names, filename='tsne.pdf'):
        tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=300)
        features_tsne = tsne.fit_transform(features)

        plt.figure(figsize=(10, 8))
        colors = list(mcolors.TABLEAU_COLORS.values())

        for i, class_name in enumerate(class_names):
            mask = labels == i
            plt.scatter(features_tsne[mask, 0], features_tsne[mask, 1],
                        c=[colors[i]], label=class_name, s=50, alpha=0.6)

        plt.xticks(fontsize=12)
        plt.yticks(fontsize=12)
        plt.xlabel('t-SNE 1', fontsize=14)
        plt.ylabel('t-SNE 2', fontsize=14)
        plt.legend(loc='upper right', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.close()

    @staticmethod
    def plot_training_metrics(history, filename='training_metrics.pdf'):
        epochs = range(1, len(history['train_loss']) + 1)

        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

        ax1.plot(epochs, history['train_loss'], 'b-', label='Train')
        ax1.plot(epochs, history['val_loss'], 'r-', label='Validation')
        ax1.set_title('Loss Curve')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.plot(epochs, history['train_acc'], 'b-', label='Train')
        ax2.plot(epochs, history['val_acc'], 'r-', label='Validation')
        ax2.set_title('Accuracy Curve')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        ax3.plot(epochs, history['train_f1'], 'b-', label='Train')
        ax3.plot(epochs, history['val_f1'], 'r-', label='Validation')
        ax3.set_title('F1 Score Curve')
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('F1 Score')
        ax3.legend()
        ax3.grid(True, alpha=0.3)

        ax4.plot(epochs, history['lr'], 'g-', label='Learning Rate')
        ax4.set_title('Learning Rate Schedule')
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('Learning Rate')
        ax4.legend()
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.close()

# ----------------- 知识蒸馏损失 -----------------
class DistillationLoss(nn.Module):
    def __init__(self, temperature=3.0, alpha=0.7, mode='features'):
        super().__init__()
        self.T = temperature
        self.alpha = alpha
        self.mode = mode
        self.ce_loss = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, student_outputs, teacher_outputs, targets):
        # 硬目标损失
        hard_loss = self.ce_loss(student_outputs['logits'], targets)

        # 软目标损失
        if self.mode == 'logits':
            soft_loss = F.kl_div(
                F.log_softmax(student_outputs['logits']/self.T, dim=1),
                F.softmax(teacher_outputs['logits']/self.T, dim=1),
                reduction='batchmean'
            ) * (self.T ** 2)
        else:  # 特征蒸馏
            soft_loss = 0
            for layer in student_outputs['features']:
                s_feat = student_outputs['features'][layer]
                t_feat = teacher_outputs['features'][layer]
                # 对特征进行L2归一化后再计算MSE损失
                s_feat = F.normalize(s_feat, p=2, dim=1)
                t_feat = F.normalize(t_feat, p=2, dim=1)
                soft_loss += F.mse_loss(s_feat, t_feat)

        return self.alpha * soft_loss + (1 - self.alpha) * hard_loss

# ----------------- TinyViT模型包装器（支持特征提取）-----------------
class TinyViTWrapper(nn.Module):
    def __init__(self, model_name='tiny_vit_5m_224', num_classes=6, pretrained=True, 
                 pretrained_path=None, distill_layers=None):
        super().__init__()
        self.model = tiny_vit_5m_224(pretrained=pretrained, num_classes=num_classes)

        if pretrained_path:
            state_dict = torch.load(pretrained_path, weights_only=True)
            self.model.load_state_dict(state_dict, strict=False)

        self.distill_layers = distill_layers or [
            f'layers.{len(self.model.layers)-3}.blocks.{i}' 
            for i in range(self.model.depths[-3])
        ]

        self.original_head = self.model.head
        self.model.head = nn.Linear(self.model.head.in_features, num_classes)

        # Get actual resolutions from model
        self.layer_resolutions = self._get_actual_resolutions()

    def _get_actual_resolutions(self):
        """Get actual resolutions from model layers"""
        resolutions = []
        # Get patch embed resolution
        H = self.model.patch_embed.patches_resolution[0]
        W = self.model.patch_embed.patches_resolution[1]
        resolutions.append((H, W))

        # Get resolutions from layers
        for layer in self.model.layers:
            if hasattr(layer, 'blocks'):
                if len(layer.blocks) > 0 and hasattr(layer.blocks[0], 'input_resolution'):
                    H, W = layer.blocks[0].input_resolution
                    resolutions.append((H, W))
                else:
                    resolutions.append(resolutions[-1])
            else:
                resolutions.append(resolutions[-1])

        return resolutions[:len(self.model.layers)+1]

    def forward(self, x, return_features=False):
        if not return_features:
            return {
                'logits': self.model(x),
                'features': None
            }

        features = {}
        B = x.shape[0]

        # 1. Patch embedding
        x = self.model.patch_embed(x)
        C, H, W = x.shape[1], x.shape[2], x.shape[3]

        # Check initial resolution
        if (H, W) != self.layer_resolutions[0]:
            raise ValueError(
                f"Initial resolution mismatch: expected {self.layer_resolutions[0]}, got {(H, W)}"
            )

        # Convert to sequence format (B, L, C)
        x = x.flatten(2).permute(0, 2, 1)  # (B, C, H*W) -> (B, H*W, C)

        # 2. Process through layers
        for layer_idx, (layer, (H_layer, W_layer)) in enumerate(zip(self.model.layers, self.layer_resolutions[1:])):
            # Handle downsampling layers
            if hasattr(layer, 'downsample') and layer.downsample is not None:
                x = layer.downsample(x)
                H, W = H_layer, W_layer
                continue

            if hasattr(layer, 'blocks'):
                for block_idx, block in enumerate(layer.blocks):
                    # Ensure input shape is correct
                    if hasattr(block, 'input_resolution'):
                        expected_H, expected_W = block.input_resolution
                        current_L = x.shape[1]
                        if current_L != expected_H * expected_W:
                            # Reshape to spatial format for interpolation
                            x = x.permute(0, 2, 1).reshape(B, -1, H, W)
                            x = F.interpolate(x, size=(expected_H, expected_W), mode='nearest')
                            x = x.flatten(2).permute(0, 2, 1)
                            H, W = expected_H, expected_W

                    # Save current shape for recovery
                    prev_shape = x.shape

                    # Execute block forward pass
                    x = block(x)

                    # Feature collection
                    name = f'layers.{layer_idx}.blocks.{block_idx}'
                    if name in self.distill_layers:
                        features[name] = x.mean(dim=1)  # (B, C)

                    # Update channel dimension
                    C = x.shape[-1]
            else:
                x = layer(x)
                if f'layers.{layer_idx}' in self.distill_layers:
                    features[f'layers.{layer_idx}'] = x.mean(dim=1)

        # 3. Final classification
        x = x.mean(1)  # (B, C)
        x = self.model.norm_head(x)
        logits = self.model.head(x)

        return {
            'logits': logits,
            'features': features
        }

# ----------------- Swin Transformer教师模型包装器 -----------------
class SwinTeacher(nn.Module):
    def __init__(self, model_name='swin_v2_b', num_classes=6):
        super().__init__()
        # 加载预训练的Swin Transformer模型
        weights = models.Swin_V2_B_Weights.IMAGENET1K_V1
        self.model = models.swin_v2_b(weights=weights)
        
        # 替换最后的分类头
        in_features = self.model.head.in_features
        self.model.head = nn.Linear(in_features, num_classes)
        
        # 特征提取层配置 - 选择Swin Transformer的关键层
        self.feature_layers = {
            'features.0': 'stage1',  # 第一阶段输出
            'features.2': 'stage3',  # 第三阶段输出
        }
        
        # 注册hook来捕获特征
        self.features = {}
        for layer_name in self.feature_layers.values():
            self.features[layer_name] = None
        
        # 获取模型的所有模块
        self.model_dict = dict([*self.model.named_modules()])
        
        # 为每个特征层注册hook
        for name, layer_name in self.feature_layers.items():
            if name in self.model_dict:
                self.model_dict[name].register_forward_hook(
                    self._get_hook(layer_name)
                )

    def _get_hook(self, name):
        def hook(module, input, output):
            # Swin Transformer的特征输出通常是(B, H*W, C)
            # 我们取空间维度的均值作为特征表示
            if isinstance(output, tuple):
                output = output[0]  # 有些层可能返回元组
            self.features[name] = output.mean(dim=1)  # (B, C)
        return hook

    def forward(self, x, return_features=False):
        # 清空特征缓存
        for k in self.features:
            self.features[k] = None
            
        # 前向传播
        logits = self.model(x)
        
        if return_features:
            return {
                'logits': logits,
                'features': self.features
            }
        return {
            'logits': logits,
            'features': None
        }

# ----------------- 训练时间记录器 -----------------
class TimeTracker:
    def __init__(self):
        self.epoch_times = []
        self.batch_times = []
        self.start_time = None

    def epoch_start(self):
        self.start_time = time.time()

    def epoch_end(self):
        epoch_time = time.time() - self.start_time
        self.epoch_times.append(epoch_time)
        return epoch_time

    def batch_end(self):
        self.batch_times.append(time.time() - self.start_time)

    def get_stats(self):
        return {
            'total_time': sum(self.epoch_times),
            'avg_epoch_time': np.mean(self.epoch_times),
            'avg_batch_time': np.mean(self.batch_times) if self.batch_times else 0,
            'epoch_times': self.epoch_times
        }

# ----------------- 训练函数 -----------------
def train_one_epoch(student, teacher, loader, optimizer, scaler, criterion, epoch, time_tracker, config):
    student.train()
    total_loss = 0.0
    preds, labels = [], []

    time_tracker.epoch_start()
    pbar = tqdm(loader, desc=f'Epoch {epoch} Training')

    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=True):
            # 教师模型推理
            with torch.no_grad():
                teacher_outputs = teacher(images, return_features=True)

            # 学生模型推理
            student_outputs = student(images, return_features=True)

            # 计算蒸馏损失
            loss = criterion(student_outputs, teacher_outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        time_tracker.batch_end()

        total_loss += loss.item()
        batch_preds = torch.argmax(student_outputs['logits'], 1).cpu().numpy()
        batch_labels = targets.cpu().numpy()
        preds.extend(batch_preds)
        labels.extend(batch_labels)

        pbar.set_postfix({'Loss': loss.item()})

    epoch_time = time_tracker.epoch_end()
    return {
        'train_loss': total_loss / len(loader),
        'train_acc': accuracy_score(labels, preds),
        'train_f1': f1_score(labels, preds, average='macro'),
        'epoch_time': epoch_time
    }

# ----------------- 验证函数 -----------------
@torch.no_grad()
def evaluate(model, loader, criterion, epoch, is_best=False):
    model.eval()
    total_loss = 0.0
    preds, labels, features = [], [], []

    for images, targets in tqdm(loader, desc='Evaluating'):
        images = images.to(device)
        targets = targets.to(device)

        with torch.cuda.amp.autocast(enabled=True):
            outputs = model(images, return_features=True)
            loss = criterion(outputs, None, targets) 

        total_loss += loss.item()
        preds.extend(torch.argmax(outputs['logits'], 1).cpu().numpy())
        labels.extend(targets.cpu().numpy())
        if outputs['features'] is not None:
            features.extend([f.mean(dim=0).cpu().numpy() for f in outputs['features'].values()])

    metrics = {
        'val_loss': total_loss / len(loader),
        'val_acc': accuracy_score(labels, preds),
        'val_precision': precision_score(labels, preds, average='macro'),
        'val_recall': recall_score(labels, preds, average='macro'),
        'val_f1': f1_score(labels, preds, average='macro')
    }

    if is_best and features:
        idx_to_labels = np.load('idx_to_labels.npy', allow_pickle=True).item()
        class_names = [idx_to_labels[i] for i in range(len(idx_to_labels))]

        features = np.array(features)
        Visualizer.plot_tsne(features, np.array(labels), class_names, 
                 filename=f'checkpoint/best_tsne_epoch{epoch}.pdf')

        cm = confusion_matrix(labels, preds)
        Visualizer.cnf_matrix_plotter(cm, class_names, filename=f'checkpoint/best_cm_epoch{epoch}.pdf')
        print("\nClassification Report:")
        print(classification_report(labels, preds, target_names=class_names, digits=4))

    return metrics

# ----------------- 主函数 -----------------
def main():
    # 初始化配置
    config = {
        'model_name': 'tiny_vit_5m_224',
        'teacher_model': 'swin_v2_b',  # 使用Swin Transformer作为教师模型
        'num_classes': 6,
        'pretrained_path': 'tiny_vit_5m_22kto1k_distill.pth',
        'distill_mode': 'features',  # 'logits' or 'features'
        'temperature': 3.0,
        'alpha': 0.7,
        'epochs': 300,
        'batch_size': 32,
        'lr': 1e-4,
        'weight_decay': 0.05,
        'seed': 42
    }

    # 初始化wandb
    wandb.init(project='soil-classification-distill-swin', config=config)
    time_tracker = TimeTracker()

    # 设置随机种子
    random.seed(config['seed'])
    np.random.seed(config['seed'])
    torch.manual_seed(config['seed'])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config['seed'])

    # 数据加载
    train_dataset = datasets.ImageFolder('soil/train', train_transform)
    test_dataset = datasets.ImageFolder('soil/val', test_transform)

    idx_to_labels = {v: k for k, v in train_dataset.class_to_idx.items()}
    np.save('idx_to_labels.npy', idx_to_labels)

    train_loader = DataLoader(
        train_dataset, batch_size=config['batch_size'], shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=config['batch_size'],
        num_workers=4, pin_memory=True, persistent_workers=True
    )

    # 初始化模型
    student = TinyViTWrapper(
        model_name=config['model_name'],
        num_classes=len(idx_to_labels),
        pretrained_path=config['pretrained_path']
    ).to(device)

    teacher = SwinTeacher(
        model_name=config['teacher_model'],
        num_classes=len(idx_to_labels)
    ).to(device).eval()

    # 冻结教师模型
    for param in teacher.parameters():
        param.requires_grad = False

    # 计算模型统计信息
    macs, params = get_model_complexity_info(student, (3, 224, 224), as_strings=False)
    model_stats = {
        'params(M)': params / 1e6,
        'FLOPs(G)': macs / 1e9,
        'MACs(G)': macs / 1e9 * 2
    }
    print("\nStudent Model Analysis:")
    print(f"Parameters: {model_stats['params(M)']:.2f}M")
    print(f"FLOPs: {model_stats['FLOPs(G)']:.2f}G")
    
    # 计算教师模型统计信息
    t_macs, t_params = get_model_complexity_info(teacher, (3, 224, 224), as_strings=False)
    teacher_stats = {
        'teacher_params(M)': t_params / 1e6,
        'teacher_FLOPs(G)': t_macs / 1e9,
        'teacher_MACs(G)': t_macs / 1e9 * 2
    }
    print("\nTeacher Model Analysis:")
    print(f"Parameters: {teacher_stats['teacher_params(M)']:.2f}M")
    print(f"FLOPs: {teacher_stats['teacher_FLOPs(G)']:.2f}G")
    
    wandb.log({**model_stats, **teacher_stats})

    # 训练准备
    criterion = DistillationLoss(
        temperature=config['temperature'],
        alpha=config['alpha'],
        mode=config['distill_mode']
    )
    optimizer = torch.optim.AdamW(
        student.parameters(), 
        lr=config['lr'], 
        weight_decay=config['weight_decay']
    )
    scaler = GradScaler()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, 
        T_0=10, 
        T_mult=2
    )

    # 训练循环
    best_acc = 0.0
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'train_f1': [], 'val_f1': [],
        'lr': [], 'epoch_times': []
    }

    for epoch in range(1, config['epochs'] + 1):
        train_metrics = train_one_epoch(
            student, teacher, train_loader, 
            optimizer, scaler, criterion, 
            epoch, time_tracker, config
        )
        val_metrics = evaluate(student, test_loader, criterion, epoch)

        # 更新最佳模型
        if val_metrics['val_acc'] > best_acc:
            best_acc = val_metrics['val_acc']
            torch.save(student.state_dict(), f'checkpoint/best_model_{best_acc:.3f}.pth')
            _ = evaluate(student, test_loader, criterion, epoch, is_best=True)
            print(f'New best model saved with acc {best_acc:.3f}')

        # 更新历史记录
        for k in history:
            if k in train_metrics: history[k].append(train_metrics[k])
            elif k in val_metrics: history[k].append(val_metrics[k])
            elif k == 'lr': history[k].append(optimizer.param_groups[0]['lr'])

        scheduler.step()

        # 记录到wandb
        wandb.log({
            **train_metrics, **val_metrics,
            'learning_rate': optimizer.param_groups[0]['lr'],
            'epoch': epoch
        })

        # 每50个epoch保存一次训练曲线
        if epoch % 50 == 0:
            Visualizer.plot_training_metrics(history)

    # 训练结束统计
    training_stats = {
        'total_time': time_tracker.get_stats()['total_time'],
        'avg_epoch_time': time_tracker.get_stats()['avg_epoch_time'],
        'best_val_acc': best_acc,
        **model_stats,
        **teacher_stats
    }
    np.savez('checkpoint/final_stats.npz', **training_stats)

    print("\nTraining Completed with Stats:")
    print(f"Total Training Time: {training_stats['total_time']/3600:.2f} hours")
    print(f"Average Epoch Time: {training_stats['avg_epoch_time']:.2f} seconds")
    print(f"Best Validation Accuracy: {best_acc:.4f}")
    wandb.finish()

if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = True
    os.makedirs('checkpoint', exist_ok=True)
    main()



In [ ]:
'''
convext  pretrain
'''
import os
import time
import random
import itertools
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import wandb
from ptflops import get_model_complexity_info
from models.tiny_vit import tiny_vit_5m_224, tiny_vit_11m_224, tiny_vit_21m_224, tiny_vit_21m_384, tiny_vit_21m_512

# ----------------- 时间跟踪器 -----------------
class TimeTracker:
    def __init__(self):
        self.epoch_start_time = 0
        self.batch_start_time = 0
        self.epoch_times = []
        self.batch_times = []
        
    def epoch_start(self):
        self.epoch_start_time = time.time()
        
    def batch_start(self):
        self.batch_start_time = time.time()
        
    def batch_end(self):
        batch_time = time.time() - self.batch_start_time
        self.batch_times.append(batch_time)
        
    def epoch_end(self):
        epoch_time = time.time() - self.epoch_start_time
        self.epoch_times.append(epoch_time)
        return epoch_time
        
    def get_stats(self):
        return {
            'total_time': sum(self.epoch_times),
            'avg_epoch_time': sum(self.epoch_times) / len(self.epoch_times),
            'avg_batch_time': sum(self.batch_times) / len(self.batch_times) if self.batch_times else 0
        }

# ----------------- 数据增强 -----------------
class AddSoilNoise:
    def __call__(self, tensor):
        noise = torch.randn_like(tensor) * 0.03 * random.random()
        return torch.clamp(tensor + noise, 0, 1)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.2)),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ----------------- 可视化工具 -----------------
class Visualizer:
    @staticmethod
    def cnf_matrix_plotter(cm, classes, cmap=plt.cm.Blues, filename='confusion_matrix.pdf'):
        plt.figure(figsize=(8, 6))
        plt.imshow(cm, interpolation='nearest', cmap=cmap)
        plt.title('Confusion Matrix', fontsize=14, pad=20)
        plt.colorbar()

        tick_marks = np.arange(len(classes))
        plt.xticks(tick_marks, classes, rotation=45, fontsize=12)
        plt.yticks(tick_marks, classes, rotation=45, fontsize=12)

        thresh = cm.max() / 2.
        for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
            plt.text(j, i, f"{cm[i, j]}",
                     horizontalalignment="center",
                     verticalalignment="center",
                     color="white" if cm[i, j] > thresh else "black",
                     fontsize=12)

        plt.ylabel('True label', fontsize=14)
        plt.xlabel('Predicted label', fontsize=14)
        plt.tight_layout()
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.close()

    @staticmethod 
    def plot_tsne(features, labels, class_names, filename='tsne.pdf'):
        tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=300)
        features_tsne = tsne.fit_transform(features)

        plt.figure(figsize=(10, 8))
        colors = list(mcolors.TABLEAU_COLORS.values())

        for i, class_name in enumerate(class_names):
            mask = labels == i
            plt.scatter(features_tsne[mask, 0], features_tsne[mask, 1],
                        c=[colors[i]], label=class_name, s=50, alpha=0.6)

        plt.xticks(fontsize=12)
        plt.yticks(fontsize=12)
        plt.xlabel('t-SNE 1', fontsize=14)
        plt.ylabel('t-SNE 2', fontsize=14)
        plt.legend(loc='upper right', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.close()

    @staticmethod
    def plot_training_metrics(history, filename='training_metrics.pdf'):
        epochs = range(1, len(history['train_loss']) + 1)

        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

        ax1.plot(epochs, history['train_loss'], 'b-', label='Train')
        ax1.plot(epochs, history['val_loss'], 'r-', label='Validation')
        ax1.set_title('Loss Curve')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.plot(epochs, history['train_acc'], 'b-', label='Train')
        ax2.plot(epochs, history['val_acc'], 'r-', label='Validation')
        ax2.set_title('Accuracy Curve')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        ax3.plot(epochs, history['train_f1'], 'b-', label='Train')
        ax3.plot(epochs, history['val_f1'], 'r-', label='Validation')
        ax3.set_title('F1 Score Curve')
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('F1 Score')
        ax3.legend()
        ax3.grid(True, alpha=0.3)

        ax4.plot(epochs, history['lr'], 'g-', label='Learning Rate')
        ax4.set_title('Learning Rate Schedule')
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('Learning Rate')
        ax4.legend()
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.close()

# ----------------- 知识蒸馏损失 -----------------
class DistillationLoss(nn.Module):
    def __init__(self, temperature=3.0, alpha=0.7, mode='logits'):
        super().__init__()
        self.T = temperature
        self.alpha = alpha
        self.mode = mode
        self.ce_loss = nn.CrossEntropyLoss(label_smoothing=0.1)
        
    def forward(self, student_outputs, teacher_outputs, targets):
        # 硬目标损失
        hard_loss = self.ce_loss(student_outputs['logits'], targets)
        
        # 软目标损失
        if self.mode == 'logits':
            soft_loss = F.kl_div(
                F.log_softmax(student_outputs['logits']/self.T, dim=1),
                F.softmax(teacher_outputs['logits']/self.T, dim=1),
                reduction='batchmean'
            ) * (self.T ** 2)
        else:  # 特征蒸馏
            soft_loss = 0
            for layer in student_outputs['features']:
                s_feat = student_outputs['features'][layer]
                t_feat = teacher_outputs['features'][layer]
                # 确保特征维度匹配
                if s_feat.dim() != t_feat.dim():
                    if s_feat.dim() == 1:
                        s_feat = s_feat.unsqueeze(0)
                    if t_feat.dim() == 1:
                        t_feat = t_feat.unsqueeze(0)
                soft_loss += F.mse_loss(s_feat, t_feat)
        
        return self.alpha * soft_loss + (1 - self.alpha) * hard_loss

# ----------------- TinyViT模型包装器（支持特征提取）-----------------
class TinyViTWrapper(nn.Module):
    def __init__(self, model_name='tiny_vit_5m_224', num_classes=6, pretrained=True, 
                 pretrained_path=None, distill_layers=None):
        super().__init__()
        self.model = tiny_vit_5m_224(pretrained=pretrained, num_classes=num_classes)
        
        if pretrained_path:
            state_dict = torch.load(pretrained_path, weights_only=True)
            self.model.load_state_dict(state_dict, strict=False)
            
        self.distill_layers = distill_layers or [
            f'layers.{len(self.model.layers)-3}.blocks.{i}' 
            for i in range(self.model.depths[-3])
        ]
        
        self.original_head = self.model.head
        self.model.head = nn.Linear(self.model.head.in_features, num_classes)
        
        # Get actual resolutions from model
        self.layer_resolutions = self._get_actual_resolutions()

    def _get_actual_resolutions(self):
        """Get actual resolutions from model layers"""
        resolutions = []
        # Get patch embed resolution
        H = self.model.patch_embed.patches_resolution[0]
        W = self.model.patch_embed.patches_resolution[1]
        resolutions.append((H, W))

        # Get resolutions from layers
        for layer in self.model.layers:
            if hasattr(layer, 'blocks'):
                if len(layer.blocks) > 0 and hasattr(layer.blocks[0], 'input_resolution'):
                    H, W = layer.blocks[0].input_resolution
                    resolutions.append((H, W))
                else:
                    resolutions.append(resolutions[-1])
            else:
                resolutions.append(resolutions[-1])

        return resolutions[:len(self.model.layers)+1]

    def forward(self, x, return_features=False):
        if not return_features:
            return {
                'logits': self.model(x),
                'features': None
            }

        features = {}
        B = x.shape[0]

        # 1. Patch embedding
        x = self.model.patch_embed(x)
        C, H, W = x.shape[1], x.shape[2], x.shape[3]

        # Check initial resolution
        if (H, W) != self.layer_resolutions[0]:
            raise ValueError(
                f"Initial resolution mismatch: expected {self.layer_resolutions[0]}, got {(H, W)}"
            )

        # Convert to sequence format (B, L, C)
        x = x.flatten(2).permute(0, 2, 1)  # (B, C, H*W) -> (B, H*W, C)

        # 2. Process through layers
        for layer_idx, (layer, (H_layer, W_layer)) in enumerate(zip(self.model.layers, self.layer_resolutions[1:])):
            # Handle downsampling layers
            if hasattr(layer, 'downsample') and layer.downsample is not None:
                x = layer.downsample(x)
                H, W = H_layer, W_layer
                continue

            if hasattr(layer, 'blocks'):
                for block_idx, block in enumerate(layer.blocks):
                    # Ensure input shape is correct
                    if hasattr(block, 'input_resolution'):
                        expected_H, expected_W = block.input_resolution
                        current_L = x.shape[1]
                        if current_L != expected_H * expected_W:
                            # Reshape to spatial format for interpolation
                            x = x.permute(0, 2, 1).reshape(B, -1, H, W)
                            x = F.interpolate(x, size=(expected_H, expected_W), mode='nearest')
                            x = x.flatten(2).permute(0, 2, 1)
                            H, W = expected_H, expected_W

                    # Save current shape for recovery
                    prev_shape = x.shape

                    # Execute block forward pass
                    x = block(x)

                    # Feature collection
                    name = f'layers.{layer_idx}.blocks.{block_idx}'
                    if name in self.distill_layers:
                        features[name] = x.mean(dim=1)  # (B, C)

                    # Update channel dimension
                    C = x.shape[-1]
            else:
                x = layer(x)
                if f'layers.{layer_idx}' in self.distill_layers:
                    features[f'layers.{layer_idx}'] = x.mean(dim=1)

        # 3. Final classification
        x = x.mean(1)  # (B, C)
        x = self.model.norm_head(x)
        logits = self.model.head(x)

        return {
            'logits': logits,
            'features': features
        }

# ----------------- ConvNeXt教师模型包装器 -----------------
class ConvNeXtTeacher(nn.Module):
    def __init__(self, model_name='convnext_base', num_classes=6):
        super().__init__()
        # Load pretrained model
        self.base_model = getattr(models, model_name)(pretrained=True)
        
        # Extract components
        self.features = nn.Sequential(
            self.base_model.features[0],  # Stem
            *self.base_model.features[1:]  # All other feature layers
        )
        self.avgpool = self.base_model.avgpool
        
        # Replace final classifier
        in_features = self.base_model.classifier[2].in_features
        self.classifier = nn.Linear(in_features, num_classes)
        
        # Feature extraction points
        self.feature_layers = {
            3: None,  # After stage 1
            6: None   # After stage 2
        }
        
    def forward(self, x, return_features=False):
        features = {}
        
        # Process through feature layers
        for i, layer in enumerate(self.features):
            x = layer(x)
            
            # Store features at specified layers
            if i in self.feature_layers:
                # Ensure proper dimensions
                if x.dim() == 4:
                    features[i] = F.adaptive_avg_pool2d(x, (1, 1)).flatten(1)
                else:
                    features[i] = x

        # Final classification
        if x.dim() == 4:
            x = self.avgpool(x)
            x = torch.flatten(x, 1)
        
        x = self.classifier(x)

        return {
            'logits': x,
            'features': features if return_features else None
        }



# ----------------- 训练函数 -----------------
def train_one_epoch(student, teacher, loader, optimizer, scaler, criterion, epoch, time_tracker, config):
    student.train()
    total_loss = 0.0
    preds, labels = [], []

    time_tracker.epoch_start()
    pbar = tqdm(loader, desc=f'Epoch {epoch} Training')

    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type='cuda', enabled=True):
            # 教师模型推理
            with torch.no_grad():
                teacher_outputs = teacher(images, return_features=True)
            
            # 学生模型推理
            student_outputs = student(images, return_features=True)
            
            # 计算蒸馏损失
            loss = criterion(student_outputs, teacher_outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        time_tracker.batch_end()

        total_loss += loss.item()
        batch_preds = torch.argmax(student_outputs['logits'], 1).cpu().numpy()
        batch_labels = targets.cpu().numpy()
        preds.extend(batch_preds)
        labels.extend(batch_labels)

        pbar.set_postfix({'Loss': loss.item()})

    epoch_time = time_tracker.epoch_end()
    return {
        'train_loss': total_loss / len(loader),
        'train_acc': accuracy_score(labels, preds),
        'train_f1': f1_score(labels, preds, average='macro'),
        'epoch_time': epoch_time
    }

# ----------------- 验证函数 -----------------
@torch.no_grad()
def evaluate(model, loader, criterion, epoch, is_best=False):
    model.eval()
    total_loss = 0.0
    preds, labels, features = [], [], []

    for images, targets in tqdm(loader, desc='Evaluating'):
        images = images.to(device)
        targets = targets.to(device)

        with torch.amp.autocast(device_type='cuda', enabled=True):
            outputs = model(images, return_features=True)
            loss = criterion(outputs, None, targets) 

        total_loss += loss.item()
        preds.extend(torch.argmax(outputs['logits'], 1).cpu().numpy())
        labels.extend(targets.cpu().numpy())
        if outputs['features'] is not None:
            features.extend([f.mean(dim=0).cpu().numpy() for f in outputs['features'].values()])

    metrics = {
        'val_loss': total_loss / len(loader),
        'val_acc': accuracy_score(labels, preds),
        'val_precision': precision_score(labels, preds, average='macro'),
        'val_recall': recall_score(labels, preds, average='macro'),
        'val_f1': f1_score(labels, preds, average='macro')
    }

    if is_best and features:
        idx_to_labels = np.load('idx_to_labels.npy', allow_pickle=True).item()
        class_names = [idx_to_labels[i] for i in range(len(idx_to_labels))]

        features = np.array(features)
        Visualizer.plot_tsne(features, np.array(labels), class_names, 
                 filename=f'checkpoint/best_tsne_epoch{epoch}.pdf')

        cm = confusion_matrix(labels, preds)
        Visualizer.cnf_matrix_plotter(cm, class_names, filename=f'checkpoint/best_cm_epoch{epoch}.pdf')
        print("\nClassification Report:")
        print(classification_report(labels, preds, target_names=class_names, digits=4))

    return metrics

# ----------------- 主函数 -----------------
def main():
    # 初始化配置
    config = {
        'model_name': 'tiny_vit_5m_224',
        'teacher_model': 'convnext_base',  # 修改为ConvNeXt
        'num_classes': 6,
        'pretrained_path': 'tiny_vit_5m_22kto1k_distill.pth',
        'distill_mode': 'features',  # 'logits' or 'features'
        'temperature': 3.0,
        'alpha': 0.7,
        'epochs': 300,
        'batch_size': 32,
        'lr': 3e-4,
        'weight_decay': 0.05,
        'seed': 42
    }

    # 初始化wandb
    wandb.init(project='soil-classification-distill', config=config)
    time_tracker = TimeTracker()

    # 设置随机种子
    random.seed(config['seed'])
    np.random.seed(config['seed'])
    torch.manual_seed(config['seed'])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config['seed'])

    # 数据加载
    train_dataset = datasets.ImageFolder('soil/train', train_transform)
    test_dataset = datasets.ImageFolder('soil/val', test_transform)

    idx_to_labels = {v: k for k, v in train_dataset.class_to_idx.items()}
    np.save('idx_to_labels.npy', idx_to_labels)

    train_loader = DataLoader(
        train_dataset, batch_size=config['batch_size'], shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=config['batch_size'],
        num_workers=4, pin_memory=True, persistent_workers=True
    )

    # 初始化模型
    student = TinyViTWrapper(
        model_name=config['model_name'],
        num_classes=len(idx_to_labels),
        pretrained_path=config['pretrained_path']
    ).to(device)

    teacher = ConvNeXtTeacher(
        model_name=config['teacher_model'],
        num_classes=len(idx_to_labels)
    ).to(device).eval()

    # 冻结教师模型
    for param in teacher.parameters():
        param.requires_grad = False

    # 计算模型统计信息
    macs, params = get_model_complexity_info(student, (3, 224, 224), as_strings=False)
    model_stats = {
        'params(M)': params / 1e6,
        'FLOPs(G)': macs / 1e9,
        'MACs(G)': macs / 1e9 * 2
    }
    print("\nStudent Model Analysis:")
    print(f"Parameters: {model_stats['params(M)']:.2f}M")
    print(f"FLOPs: {model_stats['FLOPs(G)']:.2f}G")
    wandb.log(model_stats)

    # 训练准备
    criterion = DistillationLoss(
        temperature=config['temperature'],
        alpha=config['alpha'],
        mode=config['distill_mode']
    )
    optimizer = torch.optim.AdamW(
        student.parameters(), 
        lr=config['lr'], 
        weight_decay=config['weight_decay']
    )
    scaler = GradScaler()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, 
        T_0=10, 
        T_mult=2
    )

    # 训练循环
    best_acc = 0.0
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'train_f1': [], 'val_f1': [],
        'lr': [], 'epoch_times': []
    }

    for epoch in range(1, config['epochs'] + 1):
        train_metrics = train_one_epoch(
            student, teacher, train_loader, 
            optimizer, scaler, criterion, 
            epoch, time_tracker, config
        )
        val_metrics = evaluate(student, test_loader, criterion, epoch)

        # 更新最佳模型
        if val_metrics['val_acc'] > best_acc:
            best_acc = val_metrics['val_acc']
            torch.save(student.state_dict(), f'checkpoint/best_model_{best_acc:.3f}.pth')
            _ = evaluate(student, test_loader, criterion, epoch, is_best=True)
            print(f'New best model saved with acc {best_acc:.3f}')

        # 更新历史记录
        for k in history:
            if k in train_metrics: history[k].append(train_metrics[k])
            elif k in val_metrics: history[k].append(val_metrics[k])
            elif k == 'lr': history[k].append(optimizer.param_groups[0]['lr'])

        scheduler.step()

        # 记录到wandb
        wandb.log({
            **train_metrics, **val_metrics,
            'learning_rate': optimizer.param_groups[0]['lr'],
            'epoch': epoch
        })

        # 每50个epoch保存一次训练曲线
        if epoch % 50 == 0:
            Visualizer.plot_training_metrics(history)

    # 训练结束统计
    training_stats = {
        'total_time': time_tracker.get_stats()['total_time'],
        'avg_epoch_time': time_tracker.get_stats()['avg_epoch_time'],
        'best_val_acc': best_acc,
        **model_stats
    }
    np.savez('checkpoint/final_stats.npz', **training_stats)

    print("\nTraining Completed with Stats:")
    print(f"Total Training Time: {training_stats['total_time']/3600:.2f} hours")
    print(f"Average Epoch Time: {training_stats['avg_epoch_time']:.2f} seconds")
    print(f"Best Validation Accuracy: {best_acc:.4f}")
    wandb.finish()

if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = True
    os.makedirs('checkpoint', exist_ok=True)
    main()


In [ ]:
'''
vit
'''
import os
import time
import random
import itertools
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import wandb
from ptflops import get_model_complexity_info
from models.tiny_vit import tiny_vit_5m_224, tiny_vit_11m_224, tiny_vit_21m_224, tiny_vit_21m_384, tiny_vit_21m_512

# ----------------- 数据增强 -----------------
class AddSoilNoise:
    def __call__(self, tensor):
        noise = torch.randn_like(tensor) * 0.03 * random.random()
        return torch.clamp(tensor + noise, 0, 1)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    AddSoilNoise(),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.2)),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ----------------- 可视化工具 -----------------
class Visualizer:
    @staticmethod
    def cnf_matrix_plotter(cm, classes, cmap=plt.cm.Blues, filename='confusion_matrix.pdf'):
        plt.figure(figsize=(8, 6))
        plt.imshow(cm, interpolation='nearest', cmap=cmap)
        plt.title('Confusion Matrix', fontsize=14, pad=20)
        plt.colorbar()

        tick_marks = np.arange(len(classes))
        plt.xticks(tick_marks, classes, rotation=45, fontsize=12)
        plt.yticks(tick_marks, classes, rotation=45, fontsize=12)

        thresh = cm.max() / 2.
        for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
            plt.text(j, i, f"{cm[i, j]}",
                     horizontalalignment="center",
                     verticalalignment="center",
                     color="white" if cm[i, j] > thresh else "black",
                     fontsize=12)

        plt.ylabel('True label', fontsize=14)
        plt.xlabel('Predicted label', fontsize=14)
        plt.tight_layout()
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.close()

    @staticmethod 
    def plot_tsne(features, labels, class_names, filename='tsne.pdf'):
        tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=300)
        features_tsne = tsne.fit_transform(features)

        plt.figure(figsize=(10, 8))
        colors = list(mcolors.TABLEAU_COLORS.values())

        for i, class_name in enumerate(class_names):
            mask = labels == i
            plt.scatter(features_tsne[mask, 0], features_tsne[mask, 1],
                        c=[colors[i]], label=class_name, s=50, alpha=0.6)

        plt.xticks(fontsize=12)
        plt.yticks(fontsize=12)
        plt.xlabel('t-SNE 1', fontsize=14)
        plt.ylabel('t-SNE 2', fontsize=14)
        plt.legend(loc='upper right', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.close()

    @staticmethod
    def plot_training_metrics(history, filename='training_metrics.pdf'):
        epochs = range(1, len(history['train_loss']) + 1)

        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

        ax1.plot(epochs, history['train_loss'], 'b-', label='Train')
        ax1.plot(epochs, history['val_loss'], 'r-', label='Validation')
        ax1.set_title('Loss Curve')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.plot(epochs, history['train_acc'], 'b-', label='Train')
        ax2.plot(epochs, history['val_acc'], 'r-', label='Validation')
        ax2.set_title('Accuracy Curve')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        ax3.plot(epochs, history['train_f1'], 'b-', label='Train')
        ax3.plot(epochs, history['val_f1'], 'r-', label='Validation')
        ax3.set_title('F1 Score Curve')
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('F1 Score')
        ax3.legend()
        ax3.grid(True, alpha=0.3)

        ax4.plot(epochs, history['lr'], 'g-', label='Learning Rate')
        ax4.set_title('Learning Rate Schedule')
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('Learning Rate')
        ax4.legend()
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.close()

# ----------------- 知识蒸馏损失 -----------------
class DistillationLoss(nn.Module):
    def __init__(self, temperature=3.0, alpha=0.7, mode='logits'):
        super().__init__()
        self.T = temperature
        self.alpha = alpha
        self.mode = mode
        self.ce_loss = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, student_outputs, teacher_outputs, targets):
        # 硬目标损失
        hard_loss = self.ce_loss(student_outputs['logits'], targets)

        # 软目标损失
        if self.mode == 'logits':
            soft_loss = F.kl_div(
                F.log_softmax(student_outputs['logits']/self.T, dim=1),
                F.softmax(teacher_outputs['logits']/self.T, dim=1),
                reduction='batchmean'
            ) * (self.T ** 2)
        else:  # 特征蒸馏
            soft_loss = 0
            for layer in student_outputs['features']:
                s_feat = student_outputs['features'][layer]
                t_feat = teacher_outputs['features'][layer]
                soft_loss += F.mse_loss(s_feat, t_feat)

        return self.alpha * soft_loss + (1 - self.alpha) * hard_loss

# ----------------- TinyViT模型包装器（支持特征提取）-----------------
class TinyViTWrapper(nn.Module):
    def __init__(self, model_name='tiny_vit_5m_224', num_classes=6, pretrained=True, 
                 pretrained_path=None, distill_layers=None):
        super().__init__()
        self.model = tiny_vit_5m_224(pretrained=pretrained, num_classes=num_classes)

        if pretrained_path:
            state_dict = torch.load(pretrained_path, weights_only=True)
            self.model.load_state_dict(state_dict, strict=False)

        self.distill_layers = distill_layers or [
            f'layers.{len(self.model.layers)-3}.blocks.{i}' 
            for i in range(self.model.depths[-3])
        ]

        self.original_head = self.model.head
        self.model.head = nn.Linear(self.model.head.in_features, num_classes)

        # Get actual resolutions from model
        self.layer_resolutions = self._get_actual_resolutions()

    def _get_actual_resolutions(self):
        """Get actual resolutions from model layers"""
        resolutions = []
        # Get patch embed resolution
        H = self.model.patch_embed.patches_resolution[0]
        W = self.model.patch_embed.patches_resolution[1]
        resolutions.append((H, W))

        # Get resolutions from layers
        for layer in self.model.layers:
            if hasattr(layer, 'blocks'):
                if len(layer.blocks) > 0 and hasattr(layer.blocks[0], 'input_resolution'):
                    H, W = layer.blocks[0].input_resolution
                    resolutions.append((H, W))
                else:
                    resolutions.append(resolutions[-1])
            else:
                resolutions.append(resolutions[-1])

        return resolutions[:len(self.model.layers)+1]

    def forward(self, x, return_features=False):
        if not return_features:
            return {
                'logits': self.model(x),
                'features': None
            }

        features = {}
        B = x.shape[0]

        # 1. Patch embedding
        x = self.model.patch_embed(x)
        C, H, W = x.shape[1], x.shape[2], x.shape[3]

        # Check initial resolution
        if (H, W) != self.layer_resolutions[0]:
            raise ValueError(
                f"Initial resolution mismatch: expected {self.layer_resolutions[0]}, got {(H, W)}"
            )

        # Convert to sequence format (B, L, C)
        x = x.flatten(2).permute(0, 2, 1)  # (B, C, H*W) -> (B, H*W, C)

        # 2. Process through layers
        for layer_idx, (layer, (H_layer, W_layer)) in enumerate(zip(self.model.layers, self.layer_resolutions[1:])):
            # Handle downsampling layers
            if hasattr(layer, 'downsample') and layer.downsample is not None:
                x = layer.downsample(x)
                H, W = H_layer, W_layer
                continue

            if hasattr(layer, 'blocks'):
                for block_idx, block in enumerate(layer.blocks):
                    # Ensure input shape is correct
                    if hasattr(block, 'input_resolution'):
                        expected_H, expected_W = block.input_resolution
                        current_L = x.shape[1]
                        if current_L != expected_H * expected_W:
                            # Reshape to spatial format for interpolation
                            x = x.permute(0, 2, 1).reshape(B, -1, H, W)
                            x = F.interpolate(x, size=(expected_H, expected_W), mode='nearest')
                            x = x.flatten(2).permute(0, 2, 1)
                            H, W = expected_H, expected_W

                    # Save current shape for recovery
                    prev_shape = x.shape

                    # Execute block forward pass
                    x = block(x)

                    # Feature collection
                    name = f'layers.{layer_idx}.blocks.{block_idx}'
                    if name in self.distill_layers:
                        features[name] = x.mean(dim=1)  # (B, C)

                    # Update channel dimension
                    C = x.shape[-1]
            else:
                x = layer(x)
                if f'layers.{layer_idx}' in self.distill_layers:
                    features[f'layers.{layer_idx}'] = x.mean(dim=1)

        # 3. Final classification
        x = x.mean(1)  # (B, C)
        x = self.model.norm_head(x)
        logits = self.model.head(x)

        return {
            'logits': logits,
            'features': features
        }

# ----------------- ViT教师模型包装器 -----------------
# ----------------- ViT教师模型包装器 -----------------
class ViTTeacher(nn.Module):
    def __init__(self, model_name='vit_l_16', num_classes=6):
        super().__init__()
        self.model = models.vit_l_16(weights=models.ViT_L_16_Weights.IMAGENET1K_SWAG_LINEAR_V1)
        
        # 替换最后的分类头
        in_features = self.model.heads.head.in_features
        self.model.heads.head = nn.Linear(in_features, num_classes)
        
        # 特征提取层配置 - 使用ViT的实际层名称
        self.feature_layers = [
            'encoder.layers.encoder_layer_11',  # 最后一层
            'encoder.layers.encoder_layer_5'    # 中间层
        ]
        
        # 注册hook来捕获特征
        self.features = {}
        for layer_name in self.feature_layers:
            # 获取模型的所有模块
            modules = dict([*self.model.named_modules()])
            if layer_name not in modules:
                raise ValueError(f"Layer {layer_name} not found in model. Available layers: {list(modules.keys())}")
            
            layer = modules[layer_name]
            layer.register_forward_hook(self._get_hook(layer_name))

    def _get_hook(self, name):
        def hook(module, input, output):
            # ViT的输出通常是(B, N+1, C)，其中N是patch数量
            # 我们取类token（第一个token）作为特征表示
            self.features[name] = output[:, 0]  # 取类token (B, C)
        return hook

    def forward(self, x, return_features=False):
        self.features = {}  # 清空特征缓存
        
        if return_features:
            logits = self.model(x)
            return {
                'logits': logits,
                'features': self.features
            }
        return {
            'logits': self.model(x),
            'features': None
        }

# ----------------- 训练时间记录器 -----------------
class TimeTracker:
    def __init__(self):
        self.epoch_times = []
        self.batch_times = []
        self.start_time = None

    def epoch_start(self):
        self.start_time = time.time()

    def epoch_end(self):
        epoch_time = time.time() - self.start_time
        self.epoch_times.append(epoch_time)
        return epoch_time

    def batch_end(self):
        self.batch_times.append(time.time() - self.start_time)

    def get_stats(self):
        return {
            'total_time': sum(self.epoch_times),
            'avg_epoch_time': np.mean(self.epoch_times),
            'avg_batch_time': np.mean(self.batch_times) if self.batch_times else 0,
            'epoch_times': self.epoch_times
        }

# ----------------- 训练函数 -----------------
def train_one_epoch(student, teacher, loader, optimizer, scaler, criterion, epoch, time_tracker, config):
    student.train()
    total_loss = 0.0
    preds, labels = [], []

    time_tracker.epoch_start()
    pbar = tqdm(loader, desc=f'Epoch {epoch} Training')

    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=True):
            # 教师模型推理
            with torch.no_grad():
                teacher_outputs = teacher(images, return_features=True)

            # 学生模型推理
            student_outputs = student(images, return_features=True)

            # 计算蒸馏损失
            loss = criterion(student_outputs, teacher_outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        time_tracker.batch_end()

        total_loss += loss.item()
        batch_preds = torch.argmax(student_outputs['logits'], 1).cpu().numpy()
        batch_labels = targets.cpu().numpy()
        preds.extend(batch_preds)
        labels.extend(batch_labels)

        pbar.set_postfix({'Loss': loss.item()})

    epoch_time = time_tracker.epoch_end()
    return {
        'train_loss': total_loss / len(loader),
        'train_acc': accuracy_score(labels, preds),
        'train_f1': f1_score(labels, preds, average='macro'),
        'epoch_time': epoch_time
    }

# ----------------- 验证函数 -----------------
@torch.no_grad()
def evaluate(model, loader, criterion, epoch, is_best=False):
    model.eval()
    total_loss = 0.0
    preds, labels, features = [], [], []

    for images, targets in tqdm(loader, desc='Evaluating'):
        images = images.to(device)
        targets = targets.to(device)

        with torch.cuda.amp.autocast(enabled=True):
            outputs = model(images, return_features=True)
            loss = criterion(outputs, None, targets) 

        total_loss += loss.item()
        preds.extend(torch.argmax(outputs['logits'], 1).cpu().numpy())
        labels.extend(targets.cpu().numpy())
        if outputs['features'] is not None:
            features.extend([f.mean(dim=0).cpu().numpy() for f in outputs['features'].values()])

    metrics = {
        'val_loss': total_loss / len(loader),
        'val_acc': accuracy_score(labels, preds),
        'val_precision': precision_score(labels, preds, average='macro'),
        'val_recall': recall_score(labels, preds, average='macro'),
        'val_f1': f1_score(labels, preds, average='macro')
    }

    if is_best and features:
        idx_to_labels = np.load('idx_to_labels.npy', allow_pickle=True).item()
        class_names = [idx_to_labels[i] for i in range(len(idx_to_labels))]

        features = np.array(features)
        Visualizer.plot_tsne(features, np.array(labels), class_names, 
                 filename=f'checkpoint/best_tsne_epoch{epoch}.pdf')

        cm = confusion_matrix(labels, preds)
        Visualizer.cnf_matrix_plotter(cm, class_names, filename=f'checkpoint/best_cm_epoch{epoch}.pdf')
        print("\nClassification Report:")
        print(classification_report(labels, preds, target_names=class_names, digits=4))

    return metrics

# ----------------- 主函数 -----------------
def main():
    # 初始化配置
    config = {
        'model_name': 'tiny_vit_5m_224',
        'teacher_model': 'vit_l_16',  # 使用ViT作为教师模型
        'num_classes': 6,
        'pretrained_path': 'tiny_vit_5m_22kto1k_distill.pth',
        'distill_mode': 'features',  # 'logits' or 'features'
        'temperature': 3.0,
        'alpha': 0.7,
        'epochs': 300,
        'batch_size': 32,
        'lr': 1e-4,
        'weight_decay': 0.05,
        'seed': 42
    }

    # 初始化wandb
    wandb.init(project='soil-classification-distill-vit', config=config)
    time_tracker = TimeTracker()

    # 设置随机种子
    random.seed(config['seed'])
    np.random.seed(config['seed'])
    torch.manual_seed(config['seed'])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config['seed'])

    # 数据加载
    train_dataset = datasets.ImageFolder('soil/train', train_transform)
    test_dataset = datasets.ImageFolder('soil/val', test_transform)

    idx_to_labels = {v: k for k, v in train_dataset.class_to_idx.items()}
    np.save('idx_to_labels.npy', idx_to_labels)

    train_loader = DataLoader(
        train_dataset, batch_size=config['batch_size'], shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=config['batch_size'],
        num_workers=4, pin_memory=True, persistent_workers=True
    )

    # 初始化模型
    student = TinyViTWrapper(
        model_name=config['model_name'],
        num_classes=len(idx_to_labels),
        pretrained_path=config['pretrained_path']
    ).to(device)

    teacher = ViTTeacher(
        model_name=config['teacher_model'],
        num_classes=len(idx_to_labels)
    ).to(device).eval()

    # 冻结教师模型
    for param in teacher.parameters():
        param.requires_grad = False

    # 计算模型统计信息
    macs, params = get_model_complexity_info(student, (3, 224, 224), as_strings=False)
    model_stats = {
        'params(M)': params / 1e6,
        'FLOPs(G)': macs / 1e9,
        'MACs(G)': macs / 1e9 * 2
    }
    print("\nStudent Model Analysis:")
    print(f"Parameters: {model_stats['params(M)']:.2f}M")
    print(f"FLOPs: {model_stats['FLOPs(G)']:.2f}G")
    
    # 计算教师模型统计信息
    t_macs, t_params = get_model_complexity_info(teacher, (3, 224, 224), as_strings=False)
    teacher_stats = {
        'teacher_params(M)': t_params / 1e6,
        'teacher_FLOPs(G)': t_macs / 1e9,
        'teacher_MACs(G)': t_macs / 1e9 * 2
    }
    print("\nTeacher Model Analysis:")
    print(f"Parameters: {teacher_stats['teacher_params(M)']:.2f}M")
    print(f"FLOPs: {teacher_stats['teacher_FLOPs(G)']:.2f}G")
    
    wandb.log({**model_stats, **teacher_stats})

    # 训练准备
    criterion = DistillationLoss(
        temperature=config['temperature'],
        alpha=config['alpha'],
        mode=config['distill_mode']
    )
    optimizer = torch.optim.AdamW(
        student.parameters(), 
        lr=config['lr'], 
        weight_decay=config['weight_decay']
    )
    scaler = GradScaler()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, 
        T_0=10, 
        T_mult=2
    )

    # 训练循环
    best_acc = 0.0
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'train_f1': [], 'val_f1': [],
        'lr': [], 'epoch_times': []
    }

    for epoch in range(1, config['epochs'] + 1):
        train_metrics = train_one_epoch(
            student, teacher, train_loader, 
            optimizer, scaler, criterion, 
            epoch, time_tracker, config
        )
        val_metrics = evaluate(student, test_loader, criterion, epoch)

        # 更新最佳模型
        if val_metrics['val_acc'] > best_acc:
            best_acc = val_metrics['val_acc']
            torch.save(student.state_dict(), f'checkpoint/best_model_{best_acc:.3f}.pth')
            _ = evaluate(student, test_loader, criterion, epoch, is_best=True)
            print(f'New best model saved with acc {best_acc:.3f}')

        # 更新历史记录
        for k in history:
            if k in train_metrics: history[k].append(train_metrics[k])
            elif k in val_metrics: history[k].append(val_metrics[k])
            elif k == 'lr': history[k].append(optimizer.param_groups[0]['lr'])

        scheduler.step()

        # 记录到wandb
        wandb.log({
            **train_metrics, **val_metrics,
            'learning_rate': optimizer.param_groups[0]['lr'],
            'epoch': epoch
        })

        # 每50个epoch保存一次训练曲线
        if epoch % 50 == 0:
            Visualizer.plot_training_metrics(history)

    # 训练结束统计
    training_stats = {
        'total_time': time_tracker.get_stats()['total_time'],
        'avg_epoch_time': time_tracker.get_stats()['avg_epoch_time'],
        'best_val_acc': best_acc,
        **model_stats,
        **teacher_stats
    }
    np.savez('checkpoint/final_stats.npz', **training_stats)

    print("\nTraining Completed with Stats:")
    print(f"Total Training Time: {training_stats['total_time']/3600:.2f} hours")
    print(f"Average Epoch Time: {training_stats['avg_epoch_time']:.2f} seconds")
    print(f"Best Validation Accuracy: {best_acc:.4f}")
    wandb.finish()

if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = True
    os.makedirs('checkpoint', exist_ok=True)
    main()
